In [1]:
# Statistics

In [2]:
# Import relevant Python packages:
import xarray as xr
import numpy as np
import polars as pl
import matplotlib.pyplot as plt

# Following pip installation as shown on the LT Toolbox github:
import lt_toolbox as ltt
import gsw.density as density

# Searching the OceanDataCatalog dataset

from OceanDataStore import OceanDataCatalog

catalog = OceanDataCatalog(catalog_name="noc-model-stac")

#catalog.available_collections

#Era 5 probs the best here

#Searching the ERA5 collection
#catalog.search(collection='noc-npd-era5')
#catalog.available_items

t1 = catalog.open_dataset(id= 'noc-npd-era5/npd-eorca1-era5v1/gn/T1y',
                          start_datetime='2000-01',
                          end_datetime='2010-12',
                          )

#t1


d1 = catalog.open_dataset(id= 'noc-npd-era5/npd-eorca1-era5v1/gn/domain/domain_cfg',
                          )

d1

  2026-03-20T16:30:15.301675Z  WARN aws_config::imds::region: failed to load region from IMDS, err: failed to load IMDS session token: dispatch failure: timeout: client error (Connect): HTTP connect timeout occurred after 1s: timed out (FailedToLoadToken(FailedToLoadToken { source: DispatchFailure(DispatchFailure { source: ConnectorError { kind: Timeout, source: hyper_util::client::legacy::Error(Connect, HttpTimeoutError { kind: "HTTP connect", duration: 1s }), connection: Unknown } }) }))
    at /root/.cargo/registry/src/index.crates.io-1949cf8c6b5b557f/aws-config-1.8.12/src/imds/region.rs:66

  2026-03-20T16:30:19.380482Z  WARN aws_config::imds::region: failed to load region from IMDS, err: failed to load IMDS session token: dispatch failure: timeout: client error (Connect): HTTP connect timeout occurred after 1s: timed out (FailedToLoadToken(FailedToLoadToken { source: DispatchFailure(DispatchFailure { source: ConnectorError { kind: Timeout, source: hyper_util::client::legacy::Error

<xarray.Dataset> Size: 710MB
Dimensions:       (y: 331, x: 360, nav_lev: 75)
Coordinates:
  * y             (y) int64 3kB 0 1 2 3 4 5 6 7 ... 324 325 326 327 328 329 330
  * x             (x) int64 3kB 0 1 2 3 4 5 6 7 ... 353 354 355 356 357 358 359
  * nav_lev       (nav_lev) int64 600B 0 1 2 3 4 5 6 7 ... 68 69 70 71 72 73 74
Data variables: (12/49)
    e1f           (y, x) float64 953kB dask.array<chunksize=(331, 360), meta=np.ndarray>
    bottom_level  (y, x) int32 477kB dask.array<chunksize=(331, 360), meta=np.ndarray>
    bathy_metry   (y, x) float32 477kB dask.array<chunksize=(331, 360), meta=np.ndarray>
    e2f           (y, x) float64 953kB dask.array<chunksize=(331, 360), meta=np.ndarray>
    e1u           (y, x) float64 953kB dask.array<chunksize=(331, 360), meta=np.ndarray>
    e2t           (y, x) float64 953kB dask.array<chunksize=(331, 360), meta=np.ndarray>
    ...            ...
    umask         (nav_lev, y, x) int8 9MB dask.array<chunksize=(75, 331, 360), meta=np.ndarray>
    vmaskutil     (y, x) int8 119kB dask.array<chunksize=(331, 360), meta=np.ndarray>
    top_level     (y, x) int32 477kB dask.array<chunksize=(331, 360), meta=np.ndarray>
    vmask         (nav_lev, y, x) int8 9MB dask.array<chunksize=(75, 331, 360), meta=np.ndarray>
    umaskutil     (y, x) int8 119kB dask.array<chunksize=(331, 360), meta=np.ndarray>
    wmask         (nav_lev, y, x) bool 9MB dask.array<chunksize=(75, 331, 360), meta=np.ndarray>
Attributes:
    CfgName:    UNKNOWN
    CfgIndex:   -999
    Iperio:     1
    Jperio:     0
    NFold:      1
    NFtype:     F
    VertCoord:  zps
    IsfCav:     0
    file_name:  mesh_mask.nc
    TimeStamp:  01/03/2025 22:19:49 -0000

In [3]:
# Making TrajFrames for all months from 1990 to 1999
import polars as pl
import lt_toolbox as ltt
from pathlib import Path

lon_mdl = t1.nav_lon.values
lat_mdl = t1.nav_lat.values
depth_mdl = t1.deptht.values

traj_geo_all = {}

for year in range(1990, 2000):
    for month in range(1, 13):
        mm = f"{month:02d}"
        key = f"{year}_{mm}"
        
        traj_filepath = f"/dssgfs01/scratch/emdh1n25/project_1m_RAPID/eORCA1/eORCA1_output/parquet/ORCA1_NPD_{mm}_{year}_25_1m_run.parquet"
        
        if not Path(traj_filepath).exists():
            print(f"Missing {key}, skipping")
            continue

        print(f"Processing {key}...")

        # Load
        dataset = pl.read_parquet(traj_filepath, use_pyarrow=True)

        # TrajFrame
        traj = ltt.TrajFrame(source=dataset, condense=True)

        # Time conversion (IMPORTANT: use correct year)
        traj_geo = traj.use_datetime(start_date=f'{year}-01-01', unit='s')

        # Coordinate transform
        traj_geo_interp = traj_geo.transform_trajectory_coords(
            lon=lon_mdl,
            lat=lat_mdl,
            depth=depth_mdl
        )

        # Store
        traj_geo_all[key] = traj_geo_interp

        # Free memory (very important at this scale)
        del dataset, traj, traj_geo

print("All done.")

Processing 1990_01...
Processing 1990_02...
Processing 1990_03...
Processing 1990_04...
Processing 1990_05...
Processing 1990_06...
Processing 1990_07...
Processing 1990_08...
Processing 1990_09...
Processing 1990_10...
Processing 1990_11...
Processing 1990_12...
Processing 1991_01...
Processing 1991_02...
Processing 1991_03...
Processing 1991_04...
Processing 1991_05...
Processing 1991_06...
Processing 1991_07...
Processing 1991_08...
Processing 1991_09...
Processing 1991_10...
Processing 1991_11...
Processing 1991_12...
Processing 1992_01...
Processing 1992_02...
Processing 1992_03...
Processing 1992_04...
Processing 1992_05...
Processing 1992_06...
Processing 1992_07...
Processing 1992_08...
Processing 1992_09...
Processing 1992_10...
Processing 1992_11...
Processing 1992_12...
Processing 1993_01...
Processing 1993_02...
Processing 1993_03...
Processing 1993_04...
Processing 1993_05...
Processing 1993_06...
Processing 1993_07...
Processing 1993_08...
Processing 1993_09...
Processing

In [5]:
df = traj_geo_all["1990_08"].data

df

id,lon,lat,depth,subvol,time,boxface,somxl010,hfds,thetao_con,so_abs
i64,list[f64],list[f64],list[f64],list[f64],list[datetime[μs]],list[i64],list[f64],list[f64],list[f64],list[f64]
1,"[-80.252003, -80.322168, … -80.441995]","[26.913318, 27.035595, … 26.913144]","[1.030808, 1.555855, … 5.967938]","[1615.2, 1615.2, … 1615.2]","[1990-01-01 00:00:00, 1990-01-09 01:15:12.220, … 1990-01-28 18:04:31.930]","[3, 5, … 4]","[21.48, 28.17, … 27.31]","[43.11, 24.08, … -16.49]","[30.25, 30.16, … 30.11]","[36.37, 36.39, … 36.36]"
2,"[-79.752027, -79.962186, … -80.312001]","[26.913773, 27.035942, … 26.913263]","[1.030808, 1.555855, … 5.967938]","[1615.2, 1615.2, … 1615.2]","[1990-01-01 00:00:00, 1990-01-09 01:15:12.220, … 1990-01-28 18:04:31.930]","[3, 5, … 4]","[21.48, 28.17, … 27.31]","[43.11, 24.08, … -16.49]","[30.25, 30.16, … 30.11]","[36.37, 36.39, … 36.36]"
3,"[-80.252003, -80.292086, … -80.431995]","[26.913318, 26.974453, … 26.913154]","[2.111768, 2.667682, … 14.396524]","[1669.24, 1669.24, … 1669.24]","[1990-01-01 00:00:00, 1990-01-04 10:43:05.640, … 1990-01-26 13:01:09.080]","[3, 5, … 4]","[21.48, 27.14, … 26.86]","[43.11, 34.6, … -11.95]","[30.25, 30.17, … 30.09]","[36.37, 36.39, … 36.36]"
4,"[-79.752027, -79.852107, … -80.282002]","[26.913773, 26.974865, … 26.91329]","[2.111768, 2.667682, … 14.396524]","[1669.24, 1669.24, … 1669.24]","[1990-01-01 00:00:00, 1990-01-04 10:43:05.640, … 1990-01-26 13:01:09.080]","[3, 5, … 4]","[21.48, 27.14, … 26.86]","[43.11, 34.6, … -11.95]","[30.25, 30.17, … 30.09]","[36.37, 36.39, … 36.36]"
5,"[-80.252003, -80.282062, … -80.411996]","[26.913318, 26.956984, … 26.913172]","[3.261981, 3.85628, … 17.4838]","[1747.47, 1747.47, … 1747.47]","[1990-01-01 00:00:00, 1990-01-03 06:18:27.630, … 1990-01-22 14:45:17.150]","[3, 5, … 4]","[21.48, 26.87, … 26.04]","[43.11, 37.4, … -3.53]","[30.25, 30.17, … 30.1]","[36.37, 36.38, … 36.36]"
…,…,…,…,…,…,…,…,…,…,…
26973,"[-16.248612, -16.188605]","[26.949414, 26.949376]","[434.709732, 432.876462]","[2086.33, 2086.33]","[1990-01-01 00:00:00, 1990-06-12 12:56:25.890]","[3, 4]","[15.66, 43.38]","[154.57, -19.24]","[12.19, 12.2]","[35.82, 35.82]"
26974,"[-15.748556, -15.498333, … -15.668547]","[26.949098, 27.144408, … 26.949047]","[434.709732, 436.084685, … 436.543002]","[2086.33, 2086.33, … 2086.33]","[1990-01-01 00:00:00, 1990-04-09 19:23:12.750, … 1990-06-14 13:20:23.850]","[3, 1, … 4]","[15.66, 24.0, … 42.84]","[154.57, -30.38, … -15.73]","[12.19, 12.21, … 12.2]","[35.82, 35.82, … 35.82]"
26975,"[-16.248612, -16.248612]","[26.949414, 26.949414]","[483.132751, 482.112466]","[1728.79, 1728.79]","[1990-01-01 00:00:00, 1990-06-08 17:18:02.430]","[3, 4]","[15.66, 44.29]","[154.57, -25.26]","[11.88, 11.88]","[35.81, 35.81]"
